# 02 — Phase 2: Stationary Features & Cross-Stock Transfer

Phase 1 built the measuring instrument. This notebook uses it, and the first
thing it does is **overturn the Phase 1 headline result**.

| # | Step |
|---|------|
| 1 | Bootstrap |
| 2 | Recovering the lost 77% of TCS history |
| 3 | Building stationary features |
| 4 | Scale audit — are they really dimensionless? |
| 5 | Raw vs stationary, head to head |
| 6 | **Was the Phase 1 TCS signal real?** |
| 7 | Cross-stock transfer — the actual deliverable |
| 8 | Horizon preview and the persistence trap |
| 9 | Saving Phase 2 results |

**Prerequisite:** `src/stationary.py` in place; notebooks 00 and 01 already run.


## 1 — Bootstrap


In [ ]:
import sys
from pathlib import Path

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
SRC = ROOT / 'src'
assert SRC.exists(), f'Could not find src/ at {SRC}'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

import config
from dataset import build_dataset, get_feature_columns
from evaluation import (compare_feature_sets, permutation_test,
                        walk_forward_evaluate)
from stationary import (add_stationary_features, compare_scales,
                        get_stationary_features, scale_report)

pd.set_option('display.width', 130)
plt.rcParams['figure.dpi'] = 110

def rf(n=300):
    return RandomForestClassifier(n_estimators=n,
                                  random_state=config.RANDOM_STATE, n_jobs=-1)

PHASE1 = pd.read_csv(config.REPORTS_DIR / 'phase1_reference_results.csv')
print('Phase 1 reference:')
print(PHASE1.to_string(index=False))


## 2 — Recovering the lost 77% of TCS history

Phase 1 modelled TCS on **565 rows**. That was never a data limit — it was
the news trim. Merging the 2008-2016 DJIA sentiment file forced the price
data down to the overlap window, discarding 2016-2023 entirely.

Phase 1 also showed those sentiment features are worthless (ROC-AUC 0.47,
below chance). So we were throwing away 77% of the history to keep two
columns that carry no information.

From here on there are **two datasets per stock**:

| Dataset | Rows (TCS) | Purpose |
|---|---|---|
| technical, full history | ~2,350 | all real modelling |
| sentiment, trimmed | ~565 | sentiment experiments only |


In [ ]:
DATA = {}
for key in config.list_stocks():
    d = build_dataset(key, with_sentiment=False, save=False, verbose=False)
    DATA[key] = d
    print(f"{key:10s} {len(d):5d} rows  "
          f"{d['Date'].min().date()} .. {d['Date'].max().date()}")

print()
print(f"TCS: Phase 1 used 565 rows, we now have {len(DATA['TCS'])} "
      f"({len(DATA['TCS']) / 565:.1f}x more)")


## 3 — Building stationary features

Every original price-level feature gets a dimensionless replacement:

| Original (rupees) | Stationary replacement |
|---|---|
| `SMA_20` | `close_vs_sma20` = Close/SMA20 − 1 |
| `SMA_50` | `close_vs_sma50`, `sma20_vs_sma50` |
| `EMA_20` | `close_vs_ema20` |
| `MACD` | `macd_norm` = MACD/Close |
| `Volume` | `volume_ratio_20`, `volume_zscore_60` |
| `cloud_thickness` | `cloud_thickness_norm` |
| `bb_price_vs_middle` | `bb_price_vs_middle_norm` |

Plus new families that were missing entirely: lagged returns, multi-scale
momentum, three volatility estimators, candle structure, and calendar flags.

The originals are kept in the frame so both sets can be compared on
**identical rows and identical folds**.


In [ ]:
for key in DATA:
    DATA[key] = (add_stationary_features(DATA[key], verbose=True)
                 .dropna().reset_index(drop=True))
    print(f'  {key}: {len(DATA[key])} rows after warm-up dropna\n')

RAW_COLS = get_feature_columns(DATA['TCS'], include_sentiment=False)
STAT_COLS = get_stationary_features(DATA['TCS'])
print(f'RAW feature count        : {len(RAW_COLS)}')
print(f'STATIONARY feature count : {len(STAT_COLS)}')


## 4 — Scale audit

A stationary feature should have a small magnitude. Anything with a mean in
the hundreds is still carrying price level and has not actually been
converted.


In [ ]:
rep_raw = scale_report(DATA['TCS'], RAW_COLS)
rep_stat = scale_report(DATA['TCS'], STAT_COLS)

print('RAW features still on price scale:')
print(rep_raw[rep_raw['likely_price_scale']][['mean', 'std', 'abs_mean']]
      .to_string())
print()
print(f"STATIONARY features still on price scale: "
      f"{int(rep_stat['likely_price_scale'].sum())}")
print()
print('Largest stationary magnitudes (all should be small):')
print(rep_stat.head(6)[['mean', 'std', 'min', 'max']].to_string())


### The transferability test

The direct question: does a feature mean the same thing on both stocks?

If `SMA_20` averages 1893 on TCS and 1154 on Reliance, a threshold learned
on one is meaningless on the other. This is the structural reason the old
design could not "just add a stock".


In [ ]:
cmp_raw = compare_scales(DATA['TCS'], DATA['RELIANCE'], RAW_COLS,
                         'TCS', 'RELIANCE')
cmp_stat = compare_scales(DATA['TCS'], DATA['RELIANCE'], STAT_COLS,
                          'TCS', 'RELIANCE')

print('RAW — worst offenders:')
print(cmp_raw.head(6).to_string())
print()
print('STATIONARY — worst offenders:')
print(cmp_stat.head(6).to_string())
print()
print(f"Transferable features:  RAW {cmp_raw['transferable'].mean():.0%}"
      f"   STATIONARY {cmp_stat['transferable'].mean():.0%}")


## 5 — Raw vs stationary, head to head

Identical rows, identical folds, identical model.


In [ ]:
HEAD2HEAD = {}
for key, d in DATA.items():
    sets = {'RAW (Phase 1 features)': RAW_COLS, 'STATIONARY': STAT_COLS}
    table, res = compare_feature_sets(d, sets, rf(), n_splits=5, embargo=52,
                                      verbose=False)
    HEAD2HEAD[key] = res
    print(f'=== {key}  (n={len(d)}) ===')
    print(table.to_string())
    print()


**Read this carefully.** Stationary wins on both stocks, but *both*
sit near 0.52 and neither beats its majority baseline.

That is not a failure of the stationary features. It is the next cell's
finding: one-day direction is close to unpredictable, and the Phase 1
number that suggested otherwise was an artifact.


## 6 — Was the Phase 1 TCS signal real?

Phase 1 reported TCS at **0.5931, +0.0886 over baseline, permutation-clean**.
On the full decade the same features score **0.5140, −0.0102 versus baseline**.

Either the extra history hurt, or the 2014-2016 window was unrepresentative.

The test: slide a 565-row window across the decade and run the *identical*
protocol at each position. If the edge is real it should appear everywhere.
If it was luck, only the original window will show it.


In [ ]:
full = DATA['TCS']
rows = []
for start in range(0, len(full) - 565, 200):
    w = full.iloc[start:start + 565].reset_index(drop=True)
    r = walk_forward_evaluate(w, RAW_COLS, rf(200), n_splits=5, embargo=52)
    base = r.baselines.mean().max()
    acc = r.per_fold['accuracy'].mean()
    rows.append({
        'window_start': w['Date'].min().date(),
        'window_end': w['Date'].max().date(),
        'accuracy': round(acc, 4),
        'best_baseline': round(base, 4),
        'edge': round(acc - base, 4),
    })

windows = pd.DataFrame(rows)
print(windows.to_string(index=False))
print()
print(f"Edge in the Phase 1 window : {windows.iloc[0]['edge']:+.4f}")
print(f"Mean edge, all other windows: {windows.iloc[1:]['edge'].mean():+.4f}")


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
colors = ['#F44336' if i == 0 else '#5C6BC0' for i in range(len(windows))]
ax.bar(range(len(windows)), windows['edge'], color=colors, alpha=0.85)
ax.axhline(0, color='#424242', lw=1)
ax.set_xticks(range(len(windows)))
ax.set_xticklabels([str(d)[:7] for d in windows['window_start']],
                   rotation=45, ha='right')
ax.set_ylabel('Edge over best baseline')
ax.set_title('TCS — edge by 565-row window start date\n'
             '(red = the window Phase 1 happened to use)',
             fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / 'phase1_signal_was_window_luck.png',
            dpi=150, bbox_inches='tight')
plt.show()


### Verdict

The Phase 1 window is the **only** one of ten showing a meaningful positive
edge. Every other window sits at or below its baseline.

The Phase 1 TCS result was **window luck**, not predictive power. It passed
the permutation test because permutation only detects leakage — it cannot
detect a genuinely favourable but unrepresentative slice of history.

This is the correct outcome for the project, not a setback. The Phase 1
harness was built to catch exactly this kind of false positive, and it did —
on our own headline result. That is a far stronger thing to defend in a viva
than a 59% number that quietly falls apart on more data.


## 7 — Cross-stock transfer

The actual Phase 2 deliverable, and the direct test of "adding a new stock
should just work".

Train on TCS only. Test on Reliance, which the model has **never seen**.
Zero Reliance rows in training.


In [ ]:
transfer = []
for name, cols in [('RAW', RAW_COLS), ('STATIONARY', STAT_COLS)]:
    m = rf()
    m.fit(DATA['TCS'][cols], DATA['TCS']['Target'])
    pred = m.predict(DATA['RELIANCE'][cols])
    y = DATA['RELIANCE']['Target']
    acc = accuracy_score(y, pred)
    maj = max(y.mean(), 1 - y.mean())
    transfer.append({
        'feature_set': name,
        'n_features': len(cols),
        'transfer_accuracy': round(acc, 4),
        'reliance_majority': round(maj, 4),
        'edge': round(acc - maj, 4),
    })

transfer_df = pd.DataFrame(transfer).set_index('feature_set')
print('TRAIN ON TCS -> TEST ON RELIANCE (no Reliance data in training)')
print(transfer_df.to_string())


This is the clearest result in the notebook.

Raw features transfer at **0.5013** — pure chance, below Reliance's own
majority baseline. The model learned TCS rupee levels, which mean nothing
on a different stock.

Stationary features transfer at **0.5640**, beating Reliance's majority
baseline by +4.7 points — while never having seen a single Reliance row.

For a project whose stated goal is "adding new stocks should just work",
this is the number that matters.


## 8 — Horizon preview, and a trap to avoid

Phase 3 will redesign the target. Here is a preview of why — and a warning
about how easy it is to fool yourself doing it.


In [ ]:
horizon_rows = []
for h in [1, 3, 5, 10, 20]:
    d = build_dataset('TCS', horizon=h, with_sentiment=False,
                      save=False, verbose=False)
    d = add_stationary_features(d).dropna().reset_index(drop=True)
    cols = get_stationary_features(d)
    r = walk_forward_evaluate(d, cols, rf(), n_splits=5, embargo=52)
    b = r.baselines.mean()
    horizon_rows.append({
        'horizon': h,
        'model_acc': round(r.per_fold['accuracy'].mean(), 4),
        'majority': round(b['majority'], 4),
        'persistence': round(b['persistence'], 4),
        'target_autocorr': round(d['Target'].autocorr(1), 4),
    })

h_df = pd.DataFrame(horizon_rows).set_index('horizon')
print(h_df.to_string())


### The overlapping-target trap

Model accuracy climbs with horizon: 0.52 at 1 day, 0.58 at 10 days. Tempting.

But look at the **persistence** column. It climbs far faster — 0.51 to 0.88.
And look at `target_autocorr`: 0.03 at horizon 1, 0.73 at horizon 10.

The reason is that at horizon *h*, `Target[t]` and `Target[t+1]` are computed
from windows sharing *h−1* days. Consecutive labels become near-copies of one
another, so "predict the same as yesterday" scores 0.88 without doing anything.

A longer horizon therefore does **not** give a free accuracy gain. It inflates
model and baseline together, and inflates the baseline more. Phase 3 has to
handle this with non-overlapping targets or event-based sampling — otherwise
we would be reporting exactly the kind of illusory improvement this notebook
just spent a section debunking.


## 9 — Save Phase 2 results


In [ ]:
p2 = []
for key, res in HEAD2HEAD.items():
    for label, r in res.items():
        s = r.summary()
        p2.append({
            'stock': key,
            'feature_set': label,
            'n_rows': len(DATA[key]),
            'n_features': len(r.feature_names),
            'accuracy_mean': round(s['accuracy_mean'], 4),
            'accuracy_std': round(s['accuracy_std'], 4),
            'f1_mean': round(s['f1_mean'], 4),
            'roc_auc_mean': round(s['roc_auc_mean'], 4),
            'best_baseline': round(r.baselines.mean().max(), 4),
            'edge': round(s['accuracy_mean'] - r.baselines.mean().max(), 4),
        })

phase2 = pd.DataFrame(p2)
phase2.to_csv(config.REPORTS_DIR / 'phase2_results.csv', index=False)
transfer_df.to_csv(config.REPORTS_DIR / 'phase2_transfer_results.csv')
windows.to_csv(config.REPORTS_DIR / 'phase2_window_stability.csv', index=False)

print('Saved to reports/:')
print('  phase2_results.csv')
print('  phase2_transfer_results.csv')
print('  phase2_window_stability.csv')
print()
print(phase2.to_string(index=False))


## Phase 2 checklist

- [ ] TCS technical dataset is ~2,350 rows, not 565
- [ ] Zero stationary features flagged `likely_price_scale`
- [ ] Transferable features: RAW ~50%, STATIONARY ~98%
- [ ] Window-stability chart shows the Phase 1 window as the lone outlier
- [ ] Cross-stock transfer: STATIONARY beats RAW by a wide margin
- [ ] Three CSVs written to `reports/`

### Where the project actually stands

| Claim | Status |
|---|---|
| Pipeline is leakage-free | Confirmed (permutation tests) |
| Features transfer across stocks | **Confirmed — the Phase 2 win** |
| 1-day direction is predictable | **Refuted on 10 years of data** |
| Sentiment (DJIA proxy) helps | Refuted (ROC-AUC 0.47) |

**Phase 3** must now tackle the target itself: thresholded three-class labels,
non-overlapping longer horizons, and confidence-gated predictions where the
model only commits when it has something to say.
